In [42]:
import os

# API key for NewsAPI
API_KEY = ######

print("API key loaded:", bool(API_KEY))


API key loaded: True


In [43]:
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests


assert API_KEY, "API_KEY not loaded"

# Use the same date range as price data
# Price data covers: 2026-01-20 to 2026-02-09
START = "2026-01-20"
END = "2026-02-09"

# NewsAPI configuration
NEWSAPI_ENDPOINT = "https://newsapi.org/v2/everything"

# Search queries for each ticker
QUERIES = {
    "AAPL": "Apple OR AAPL",
    "NVDA": "NVIDIA OR NVDA"
}

print("Using date range:", START, "to", END)
print("This matches the price data date range.")



Using date range: 2026-01-20 to 2026-02-09
This matches the price data date range.


In [ ]:
def fetch_articles_for_date(ticker, date_str, max_per_day=10):
    """Fetch articles for a specific date to ensure coverage across all price dates."""
    params = {
        "q": QUERIES[ticker],
        "from": date_str,
        "to": date_str,
        "language": "en",
        "sortBy": "relevancy",
        "pageSize": max_per_day,
        "page": 1,
    }
    headers = {"X-Api-Key": API_KEY}

    r = requests.get(NEWSAPI_ENDPOINT, params=params, headers=headers, timeout=30)
    if r.status_code != 200:
        print(f"  Warning: {ticker} {date_str} - {r.status_code}")
        return []

    rows = []
    articles = r.json().get("articles", [])

    for a in articles:
        published_at = a.get("publishedAt")
        if not published_at:
            continue

        dt = datetime.fromisoformat(published_at.replace("Z", "+00:00"))
        rows.append({
            "ticker": ticker,
            "date": dt.strftime("%Y-%m-%d"),
            "headline": (a.get("title") or "").strip(),
            "content": (a.get("content") or a.get("description") or "").strip(),
            "source": ((a.get("source") or {}).get("name") or "").strip(),
            "url": (a.get("url") or "").strip(),
        })

    return rows

In [ ]:
# Generate list of dates in the range
from datetime import datetime, timedelta

start_dt = datetime.fromisoformat(START)
end_dt = datetime.fromisoformat(END)
date_list = []
current = start_dt
while current <= end_dt:
    date_list.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)

print(f"Fetching articles for {len(date_list)} days: {START} to {END}")

# Fetch articles for each ticker and each date
all_rows = []
for t in ["AAPL", "NVDA"]:
    print(f"\nFetching {t}...")
    ticker_count = 0
    for date_str in date_list:
        rows = fetch_articles_for_date(ticker=t, date_str=date_str, max_per_day=10)
        ticker_count += len(rows)
        all_rows.extend(rows)
        time.sleep(0.2)  # Rate limiting
    print(f"  Total for {t}: {ticker_count} articles")

articles_raw = pd.DataFrame(all_rows)
print(f"\nTotal raw articles: {len(articles_raw)}")
print(f"Date coverage: {articles_raw['date'].nunique()} unique dates")
articles_raw.head()

In [ ]:
articles = articles_raw.copy()

# Standardize date and drop missing ticker/date
articles["date"] = pd.to_datetime(articles["date"], errors="coerce").dt.strftime("%Y-%m-%d")
articles = articles.dropna(subset=["ticker", "date"])
articles = articles[articles["ticker"].astype(str).str.len() > 0]

# Remove duplicates
if "url" in articles.columns:
    articles = articles.drop_duplicates(subset=["url"], keep="first")
articles = articles.drop_duplicates(subset=["ticker", "date", "headline"], keep="first")

# LIMIT TO 5 ARTICLES PER COMPANY PER DAY
print(f"Before limiting: {len(articles)} articles")
print(f"Date range: {articles['date'].min()} to {articles['date'].max()}")
print(f"Unique dates: {articles['date'].nunique()}")

# Sort by ticker, date
articles = articles.sort_values(["ticker", "date"]).reset_index(drop=True)

# Group by ticker and date, take first 5 per group
articles = articles.groupby(["ticker", "date"]).head(5).reset_index(drop=True)

print(f"\nAfter limiting to 5 per day: {len(articles)} articles")

# Keep only required columns
articles = articles[["ticker", "date", "headline", "content", "source"]]

print("\nArticles per ticker:")
print(articles["ticker"].value_counts())

# Show date coverage per ticker
print("\nDate coverage per ticker:")
for t in ["AAPL", "NVDA"]:
    dates = articles[articles["ticker"] == t]["date"].unique()
    print(f"  {t}: {len(dates)} dates")

articles.head()

In [47]:
# Save to data folder (relative to notebook location)
output_path = os.path.join("data", "articles.csv")
os.makedirs("data", exist_ok=True)  # Create data folder if it doesn't exist
articles.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved articles.csv to {output_path}")


Saved articles.csv to data\articles.csv


In [48]:
import yfinance as yf

In [49]:
from datetime import datetime, timedelta

TICKERS = ["AAPL", "NVDA"]

# yfinance end is often exclusive, add 1 day buffer
end_plus = (datetime.fromisoformat(END) + timedelta(days=1)).strftime("%Y-%m-%d")

data = yf.download(
    tickers=" ".join(TICKERS),
    start=START,
    end=end_plus,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    progress=False,
)

rows = []
for t in TICKERS:
    sub = data[t].reset_index()
    sub["ticker"] = t
    rows.append(sub)

prices = pd.concat(rows, ignore_index=True)

prices = prices.rename(columns={
    "Date": "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume",
})

prices["date"] = pd.to_datetime(prices["date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Drop rows missing ticker/date (cleaning requirement)
prices = prices.dropna(subset=["ticker", "date"])
prices = prices.drop_duplicates(subset=["ticker", "date"], keep="first")

prices = prices[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]].copy()

print("Price rows per ticker:")
print(prices["ticker"].value_counts())

prices.head()

Price rows per ticker:
ticker
AAPL    15
NVDA    15
Name: count, dtype: int64


Price,ticker,date,open,high,low,close,adj_close,volume
0,AAPL,2026-01-20,252.729996,254.789993,243.419998,246.699997,246.469376,80267500
1,AAPL,2026-01-21,248.699997,251.559998,245.179993,247.649994,247.418488,54641700
2,AAPL,2026-01-22,249.199997,251.000000,248.149994,248.350006,248.117844,39708300
3,AAPL,2026-01-23,247.320007,249.410004,244.679993,248.039993,247.808121,41689000
4,AAPL,2026-01-26,251.479996,256.559998,249.800003,255.410004,255.171234,55969200


In [50]:
# Check if prices exists (from Cell 7)
if 'prices' not in globals():
    raise NameError("'prices' is not defined. Please run Cell 7 first to fetch price data.")

# Save to data folder (relative to notebook location)
output_path = os.path.join("data", "prices.csv")
os.makedirs("data", exist_ok=True)  # Create data folder if it doesn't exist
prices.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved prices.csv to {output_path}")

Saved prices.csv to data\prices.csv
